# 🎙️ Audio Sentiment Analyzer — Qwen2.5-Omni-7B (T4 GPU / 4-bit Quantized)

## ⚙️ STEP 1: Install Dependencies


In [1]:
# ======================== MAINQWEN.ipynb ========================
# MAINQWEN.ipynb -> Environment setup, dependency installation, & GPU validation
# ||
# ||
# ||
# imports -> Logic Flow -> Environment and hardware validation:
# ||                 |
# ||                 |--- Install dependencies -> Fetch pip/apt-get packages
# ||                 |--- Check versions -> Verify torch & transformers
# ||                 |--- IF torch.cuda.is_available()
# ||                 |    └── Verify GPU -> Check name & VRAM capacity
# ||                 |--- Try import bitsandbytes -> Check 4-bit readiness
# ||                 |--- Try import Qwen2Audio -> Check model support
# ||                 |--- assert torch.cuda.is_available() -> Final GPU block
# ||
# ======================================================================

# ---------------------------------------------------------------
# SECTION: IMPORTS / CONSTANTS
# ---------------------------------------------------------------
!pip install -q -U 'transformers==4.52.0' accelerate
!pip install -q bitsandbytes>=0.43.0
!pip install -q librosa soundfile pydub noisereduce resampy soxr
!apt-get install -q -y ffmpeg

import importlib, sys, torch, transformers

# ---------------------------------------------------------------
# SECTION: MAIN / EXECUTION ENTRY POINT
# ---------------------------------------------------------------
print('━'*55)
print('✅ Dependency check:')
print(f'   torch version       : {torch.__version__}')
print(f'   transformers version: {transformers.__version__}')
print(f'   CUDA available      : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'   GPU                 : {torch.cuda.get_device_name(0)}')
    print(f'   VRAM                : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

try:
    import bitsandbytes as bnb
    print(f'   bitsandbytes version: {bnb.__version__}')
    print('✅ bitsandbytes (4-bit quantization) found!')
except ImportError as e:
    print(f'❌ bitsandbytes not found: {e}')
    print('   Fix: !pip install bitsandbytes>=0.43.0, then restart runtime')

try:
    from transformers import Qwen2AudioForConditionalGeneration, AutoProcessor
    print('✅ Qwen2AudioForConditionalGeneration found!')
except ImportError as e:
    print(f'❌ Import failed: {e}')
    print('   Fix: re-run this cell, then Runtime → Restart and run all')

print('━'*55)
assert torch.cuda.is_available(), '❌ No GPU — go to Runtime → Change runtime type → T4 GPU'
print('✅ All checks passed! Proceed to Step 2.')

Reason for being yanked: <none given>
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 60.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 52.5 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 45 not upgraded.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
✅ Dependency check:
   torch version       : 2.10.0+cu128
   transformers version: 4.52.0
   CUDA available      : True
   GPU                 : Tesla T4
   VRAM                : 15.6 GB
   bitsandbytes version: 0.49.2
✅ bitsandbytes (4-bit quantization) found!
✅ Qwen2AudioForConditionalGeneration found!
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
✅ All checks passed

## 📦 STEP 2: Imports & Configuration



In [2]:
# ========================  ========================
# Setup config -> load env vars, & verify GPU
# ||
# ||
# ||
# Functions/Methods -> Logic Flow -> Init & Validation:
# ||                 |
# ||                 |--- load_dotenv() -> Load env vars
# ||                 |--- os.getenv() -> Get diarizer URL
# ||                 |--- os.makedirs() -> Create output dir
# ||                 |--- IF torch.cuda.is_available()
# ||                 |    └── torch.cuda.get_device_name() -> Fetch GPU name
# ||                 |    └── torch.cuda.get_device_properties() -> Calc VRAM
# ||
# ======================================================================

# ---------------------------------------------------------------
# SECTION: IMPORTS
# ---------------------------------------------------------------
import os, re, json, time, warnings, tempfile
from pathlib import Path
from datetime import datetime
import numpy as np
import librosa
import soundfile as sf
import noisereduce as nr
from pydub import AudioSegment
import torch

# ---------------------------------------------------------------
# SECTION: MAIN / EXECUTION ENTRY POINT
# ---------------------------------------------------------------
warnings.filterwarnings('ignore')

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# ---------------------------------------------------------------
# SECTION: CONSTANTS
# ---------------------------------------------------------------
CONFIG = {
    "model_id"                 : "Qwen/Qwen2-Audio-7B-Instruct",
    "load_in_4bit"             : True,
    "bnb_4bit_compute_dtype"   : torch.bfloat16,
    "bnb_4bit_use_double_quant": True,
    "bnb_4bit_quant_type"      : "nf4",
    "target_sample_rate"       : 16000,
    "max_audio_duration"       : 25,
    "segment_overlap"          : 2,
    "noise_reduction_strength" : 0.5,
    "diarizer_url"             : os.getenv("DIARIZER_URL", "https://cristobal-lawless-colorationally.ngrok-free.dev/diarize"),
    "output_dir"               : "/content/sentiment_results",
    "save_cleaned_audio"       : True,
}

os.makedirs(CONFIG["output_dir"], exist_ok=True)

print(f'✅ Config loaded.')
print(f'   Model      : {CONFIG["model_id"]}')
print(f'   Quantized  : 4-bit NF4 (bfloat16 compute)')
print(f'   Bridge URL : {CONFIG["diarizer_url"]}')
print(f'   Output dir : {CONFIG["output_dir"]}')

if torch.cuda.is_available():
    print(f'   GPU        : {torch.cuda.get_device_name(0)}')
    print(f'   VRAM       : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print(f'   Est. usage : ~4-5 GB (7B @ 4-bit) + ~1 GB inference overhead')
else:
    print(f'   GPU        : ❌ NOT FOUND')

✅ Config loaded.
   Model      : Qwen/Qwen2-Audio-7B-Instruct
   Quantized  : 4-bit NF4 (bfloat16 compute)
   Bridge URL : https://cristobal-lawless-colorationally.ngrok-free.dev/diarize
   Output dir : /content/sentiment_results
   GPU        : Tesla T4
   VRAM       : 15.6 GB
   Est. usage : ~4-5 GB (7B @ 4-bit) + ~1 GB inference overhead


/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


In [3]:

#==============================================================================================
#FLOW
 # requests.get (requests) | Ping bridge health endpoint
  #  |
  #r.json | Display bridge status
#==============================================================================================


import requests
r = requests.get("https://cristobal-lawless-colorationally.ngrok-free.dev/health")
print(r.json())

{'status': 'ok', 'pipeline_loaded': True, 'diarization_loaded': True, 'transcription_loaded': True, 'whisper_model': 'small', 'device': 'cpu'}


## 🎛️ STEP 3: Audio Preprocessing Pipeline


In [4]:
# ======================== AudioPreprocessor ========================
# AudioPreprocessor -> Format, resample, & segment audio
# ||
# ||
# ||
# Functions/Methods -> preprocess() -> Main audio pipeline
# ||                 |
# ||                 |---> __init__() -> Init config vars
# ||                 |---> convert_to_wav() -> Input -> .wav
# ||                 |---> load_and_resample() -> Load mono @ target Hz
# ||                 |---> reduce_noise() -> Bypassed for accuracy
# ||                 |---> normalize_audio() -> Bypassed for accuracy
# ||                 |---> trim_silence() -> Bypassed for duration
# ||                 |---> segment_audio() -> Split to overlapping chunks
# ||                 |
# ||                 |---> Logic Flow -> Pipeline execution:
# ||                                  |
# ||                                  |--- IF valid audio
# ||                                  |    └── preprocess() -> Format, load, segment
# ||                                  |--- IF ext != .wav
# ||                                  |    └── convert_to_wav() -> Return temp .wav
# ||                                  |--- IF duration > max
# ||                                  |    └── segment_audio() -> Return overlapping chunks
# ||
# ======================================================================

# ---------------------------------------------------------------
# SECTION: CLASSES
# ---------------------------------------------------------------
class AudioPreprocessor:

    def __init__(self, config: dict):
        self.sr             = config["target_sample_rate"]
        self.max_duration   = config["max_audio_duration"]
        self.overlap        = config["segment_overlap"]
        self.noise_strength = config["noise_reduction_strength"]
        self.save_cleaned   = config["save_cleaned_audio"]
        self.output_dir     = config["output_dir"]

    def convert_to_wav(self, audio_path: str) -> str:
        audio_path = Path(audio_path)
        suffix = audio_path.suffix.lower()
        if suffix == '.wav':
            return str(audio_path)
        print(f'  🔄 Converting {suffix} → .wav')
        audio = AudioSegment.from_file(str(audio_path))
        tmp_wav = tempfile.mktemp(suffix='.wav')
        audio.export(tmp_wav, format='wav')
        return tmp_wav

    def load_and_resample(self, wav_path: str) -> "np.ndarray":
        audio, _ = librosa.load(wav_path, sr=self.sr, mono=True, res_type='soxr_hq')
        print(f'  📊 Loaded: {len(audio)/self.sr:.1f}s | {self.sr}Hz | mono')
        return audio

    def reduce_noise(self, audio: "np.ndarray") -> "np.ndarray":
        print(f'  🔇 Noise reduction SKIPPED (disabled to preserve original duration)')
        return audio

    def normalize_audio(self, audio: "np.ndarray") -> "np.ndarray":
        print(f'  🔊 Normalization SKIPPED (disabled to preserve original duration)')
        return audio

    def trim_silence(self, audio: "np.ndarray") -> "np.ndarray":
        print(f'  ✂️  Silence trim SKIPPED (disabled to preserve original duration)')
        return audio

    def segment_audio(self, audio: "np.ndarray") -> list:
        total_duration = len(audio) / self.sr
        segments = []

        if total_duration <= self.max_duration:
            segments.append((0.0, total_duration, audio))
            print(f'  📍 Single segment: {total_duration:.1f}s')
        else:
            max_samples     = int(self.max_duration * self.sr)
            overlap_samples = int(self.overlap * self.sr)
            step            = max_samples - overlap_samples
            start, seg_num  = 0, 1
            while start < len(audio):
                end       = min(start + max_samples, len(audio))
                start_sec = start / self.sr
                end_sec   = end   / self.sr
                segments.append((start_sec, end_sec, audio[start:end]))
                print(f'  📍 Segment {seg_num}: {start_sec:.1f}s – {end_sec:.1f}s')
                start += step
                seg_num += 1
                if end == len(audio):
                    break
        return segments

    def preprocess(self, audio_path: str) -> dict:
        print(f'\n🎛️  PREPROCESSING: {Path(audio_path).name}')
        print('─' * 50)

        wav_path          = self.convert_to_wav(audio_path)
        audio             = self.load_and_resample(wav_path)
        original_duration = len(audio) / self.sr
        audio             = self.reduce_noise(audio)
        audio             = self.normalize_audio(audio)
        audio             = self.trim_silence(audio)
        segments          = self.segment_audio(audio)

        cleaned_path = None
        if self.save_cleaned:
            stem         = Path(audio_path).stem
            cleaned_path = os.path.join(self.output_dir, f'{stem}_cleaned.wav')
            sf.write(cleaned_path, audio, self.sr)
            print(f'  💾 Cleaned audio saved: {cleaned_path}')

        print(f'  ✅ Done: {len(segments)} segment(s)')
        return {
            'original_file'        : str(audio_path),
            'cleaned_audio_path'   : cleaned_path,
            'original_duration_sec': round(original_duration, 2),
            'final_duration_sec'   : round(len(audio) / self.sr, 2),
            'sample_rate'          : self.sr,
            'num_segments'         : len(segments),
            'segments'             : segments,
        }

# ---------------------------------------------------------------
# SECTION: MAIN / EXECUTION ENTRY POINT
# ---------------------------------------------------------------
print('✅ AudioPreprocessor defined.')

✅ AudioPreprocessor defined.


## 🤖 STEP 4: Load Qwen2.5-Omni-7B Model (4-bit Quantized)


In [5]:
# ======================== Model & Processor Initialization ========================
# Model Initialization -> Loads 4-bit quantized Qwen2-Audio model and processor.
# ||
# ||
# ||
# Functions/Methods -> gc.collect() -> Free Python memory
# ||                 |
# ||                 |---> torch.cuda.empty_cache() -> Clear unused VRAM
# ||                 |---> BitsAndBytesConfig() -> Define 4-bit settings
# ||                 |---> from_pretrained() -> Load processor & model weights
# ||                 |---> eval() -> Set inference mode
# ||                 |---> torch.cuda.memory_allocated() -> Check VRAM used
# ||                 |---> torch.cuda.get_device_properties() -> Verify VRAM state
# ||
# ======================================================================

# ---------------------------------------------------------------
# SECTION: IMPORTS
# ---------------------------------------------------------------
from transformers import (
    Qwen2AudioForConditionalGeneration,  # ← changed
    AutoProcessor,                        # ← changed
    BitsAndBytesConfig,
)
import gc

# ---------------------------------------------------------------
# SECTION: MAIN / EXECUTION ENTRY POINT
# ---------------------------------------------------------------
gc.collect()
torch.cuda.empty_cache()
print(f'GPU before load: {torch.cuda.memory_allocated()/1e9:.2f} GB used')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=CONFIG['load_in_4bit'],
    bnb_4bit_compute_dtype=CONFIG['bnb_4bit_compute_dtype'],
    bnb_4bit_use_double_quant=CONFIG['bnb_4bit_use_double_quant'],
    bnb_4bit_quant_type=CONFIG['bnb_4bit_quant_type'],
)

print(f'\n⏳ Loading processor: {CONFIG["model_id"]}')
processor = AutoProcessor.from_pretrained(CONFIG['model_id'], trust_remote_code=True)  # ← changed
print('✅ Processor loaded.')

print(f'\n⏳ Loading model (7B, 4-bit NF4 quantized)...')
print('   First run: ~14 GB download, 10-20 min')
print('   Subsequent runs: ~2-4 min from cache')
model = Qwen2AudioForConditionalGeneration.from_pretrained(  # ← changed
    CONFIG['model_id'],
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    # attn_implementation removed — not supported by Qwen2-Audio
)
model.eval()

device     = next(model.parameters()).device
vram_used  = torch.cuda.memory_allocated() / 1e9
vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'\n✅ Model loaded successfully!')
print(f'   Class          : {type(model).__name__}')
print(f'   Device         : {device}')
print(f'   Quantization   : 4-bit NF4 (double quant enabled)')
print(f'   VRAM used      : {vram_used:.1f} GB / {vram_total:.1f} GB')
print(f'   VRAM available : {vram_total - vram_used:.1f} GB remaining for inference')
print(f'\n⚠️  NOTE: 4-bit models show "torch.uint8" dtype — expected, compute is bfloat16')

GPU before load: 0.00 GB used

⏳ Loading processor: Qwen/Qwen2-Audio-7B-Instruct


preprocessor_config.json:   0%|          | 0.00/342 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model-00002-of-00005.safetensors:   0%|          | 0.00/3.98G [00:00<?, ?B/s]

model-00001-of-00005.safetensors:   0%|          | 0.00/3.91G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/230 [00:00<?, ?B/s]


✅ Model loaded successfully!
   Class          : Qwen2AudioForConditionalGeneration
   Device         : cuda:0
   Quantization   : 4-bit NF4 (double quant enabled)
   VRAM used      : 6.2 GB / 15.6 GB
   VRAM available : 9.4 GB remaining for inference

⚠️  NOTE: 4-bit models show "torch.uint8" dtype — expected, compute is bfloat16


## 🧠 STEP 5: Sentiment Analysis Engine



In [20]:
import os
import re
import gc
import json
import time
import torch
import tempfile
import librosa
import soundfile as sf
from datetime import datetime
from pathlib import Path

# ======================== SentimentAnalyzer ========================
# SentimentAnalyzer -> Evaluates audio segments for emotion and sarcasm using Qwen.
# ======================================================================

class SentimentAnalyzer:

    EMOTION_LABELS = [
        'happy', 'satisfied', 'neutral', 'confused',
        'not_interested', 'bored', 'irritated',
        'frustrated', 'angry', 'sarcastic',
        'anxious', 'sad', 'excited', 'polite_but_cold'
    ]

    SYSTEM_PROMPT = (
        "You are Qwen, a virtual human developed by the Qwen Team, Alibaba Group, "
        "capable of perceiving auditory and visual inputs, as well as generating text and speech."
    )

    ANALYSIS_PROMPT = """Analyze this call centre audio clip. Listen to the actual vocal qualities and return ONLY a valid JSON object.

STEP 1 — LISTEN FIRST, then ask yourself:
- Is the tone genuinely warm OR artificially sweet/controlled?
- Is the pace natural OR deliberately slow/exaggerated?
- Is there real tension underneath a calm surface?
- Are positive words matched by positive vocal energy?

STEP 2 — SARCASM DETECTION (read carefully):
- Sarcasm = positive/polite WORDS + tense/controlled/sweet TONE
- Genuine warmth + positive words = NOT sarcasm
- Happy satisfied customer closing a call = NOT sarcasm
- Polite thank-you said with real warmth = NOT sarcasm
- Only flag sarcasm if: artificially sweet tone ON a complaint, deliberately slow pace mocking, rising terminal pitch on a statement, fake agreement ("absolutely sure" said flatly)
- When in doubt — do NOT flag sarcasm

STEP 3 — SCORING RULES (strictly follow):
- Use PRECISE decimals: 0.73, 0.41, 0.68 — NEVER round numbers like 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9
- Only ONE emotion score above 0.60 — the single most dominant emotion
- Every other emotion must be below 0.50
- Confidence must reflect genuine certainty — uncertain = 0.35 to 0.49
- sentiment_score must be precise: 0.73 not 0.8, -0.61 not -0.6

EMOTION GUIDE:
- happy: genuine laughter, rising natural energy, real warmth
- satisfied: calm resolved tone, even pace, natural flow
- neutral: flat monotone, no emotional coloring, scripted
- polite_but_cold: formal clipped politeness, no warmth underneath
- sarcastic: sweet/calm tone that does NOT match complaint content
- irritated: clipped words, slightly raised pitch, controlled anger
- frustrated: sighs, repetition stress, strained voice
- angry: loud fast sharp consonants, high energy, hard glottal stops
- anxious: fast pace, higher pitch, voice breaks
- bored: flat slow low energy, trailing off
- confused: rising pitch on statements, hesitations, slower pace
- excited: fast pace, high pitch variation, high energy
- not_interested: minimal effort, short flat responses

OUTPUT — return ONLY this JSON:
{
  "primary_emotion": "<one of the 14 labels>",
  "confidence": <precise 0.00-1.00>,
  "secondary_emotions": ["<label>"],
  "emotion_scores": { "happy": 0.0, ... },
  "vocal_characteristics": {
    "tone": "warm/cold/tense/neutral/aggressive/artificially_sweet",
    "pace": "slow/normal/fast/very_fast/exaggerated",
    "pitch_variation": "flat/low/moderate/high/sarcastic_rise",
    "energy_level": "low/medium/high",
    "stress_detected": true/false,
    "hesitations_detected": true/false,
    "voice_breaks_detected": true/false,
    "exaggerated_politeness": true/false
  },
  "call_quality_indicators": {
    "engagement_level": "disengaged/passive/engaged/highly_engaged",
    "escalation_risk": "low/medium/high",
    "resolution_likelihood": "unlikely/possible/likely"
  },
  "sarcasm_indicators": {
    "detected": true/false,
    "confidence": 0.00,
    "signals": ["<vocal signal>"]
  },
  "overall_sentiment": "positive/negative/neutral/mixed",
  "sentiment_score": -1.00 to 1.00,
  "audio_reasoning": "<2-3 sentences description>"
}"""

    def __init__(self, model, processor, config):
        self.model = model
        self.processor = processor
        self.sr = config['target_sample_rate']
        self.model_id = config['model_id']

    def _save_segment_to_temp(self, audio_array) -> str:
        tmp_path = tempfile.mktemp(suffix='.wav')
        sf.write(tmp_path, audio_array, self.sr)
        return tmp_path

    def _build_messages(self, audio_path: str) -> list:
        return [
            {
                'role': 'system',
                'content': [{'type': 'text', 'text': self.SYSTEM_PROMPT}]
            },
            {
                'role': 'user',
                'content': [
                    {'type': 'audio', 'audio_url': audio_path},
                    {'type': 'text',  'text': self.ANALYSIS_PROMPT},
                ]
            }
        ]

    def _parse_json_response(self, raw: str) -> dict:
        print(f'\n🔍 RAW MODEL OUTPUT (first 400 chars):\n{raw[:400]}')

        def python_dict_to_json(s):
            s = s.strip()
            # Fix dot-in-key-names
            s = re.sub(r'"(\w+)\.(\w+)"', r'"\1_\2"', s)
            s = re.sub(r"'(\w+)\.(\w+)'", r'"\1_\2"', s)
            # Replace single quotes with double quotes
            s = re.sub(r"'([^']*)'", r'"\1"', s)
            # Fix Python booleans and None
            s = re.sub(r'\bTrue\b', 'true', s)
            s = re.sub(r'\bFalse\b', 'false', s)
            s = re.sub(r'\bNone\b', 'null', s)
            # Fix trailing commas
            s = re.sub(r',\s*([}\]])', r'\1', s)
            return s

        cleaned = raw.strip()

        # Strategy 1: Direct parse
        try:
            return json.loads(cleaned)
        except json.JSONDecodeError:
            pass

        # Strategy 2: Strip markdown fences
        cleaned = re.sub(r'```json\s*|```\s*', '', cleaned).strip()
        try:
            return json.loads(cleaned)
        except json.JSONDecodeError:
            pass

        # Strategy 3: Python dict literal conversion
        try:
            return json.loads(python_dict_to_json(cleaned))
        except Exception:
            pass

        # Strategy 4: Find outermost JSON block
        start = cleaned.find('{')
        end = cleaned.rfind('}')
        if start != -1 and end != -1:
            attempt = cleaned[start:end+1]
            try:
                return json.loads(attempt)
            except json.JSONDecodeError:
                try:
                    return json.loads(python_dict_to_json(attempt))
                except Exception:
                    pass

        # Fallback: Manual Regex Salvage
        print('⚠️ All JSON strategies failed. Salvaging key fields...')
        result = {
            'parse_error': True,
            'primary_emotion': 'unknown',
            'overall_sentiment': 'unknown',
            'sentiment_score': 0.0,
            'confidence': 0.0,
            'emotion_scores': {},
            'sarcasm_indicators': {'detected': False, 'confidence': 0.0, 'signals': []},
        }

        em = re.search(r'"primary_emotion"\s*:\s*"([^"]+)"', raw)
        if em: result['primary_emotion'] = em.group(1)
        sc = re.search(r'"sentiment_score"\s*:\s*(-?[0-9.]+)', raw)
        if sc: result['sentiment_score'] = float(sc.group(1))
        ov = re.search(r'"overall_sentiment"\s*:\s*"([^"]+)"', raw)
        if ov: result['overall_sentiment'] = ov.group(1)

        # Normalize keys
        if 'emotion_score' in result and 'emotion_scores' not in result:
            result['emotion_scores'] = result.pop('emotion_score')

        # Ensure labels exist
        if 'emotion_scores' in result:
            for label in self.EMOTION_LABELS:
                if label not in result['emotion_scores']:
                    result['emotion_scores'][label] = 0.0

        return result

    def analyze_segment(self, audio_array, start_sec: float, end_sec: float) -> dict:
        gc.collect()
        torch.cuda.empty_cache()

        tmp_audio_path = self._save_segment_to_temp(audio_array)

        try:
            messages = self._build_messages(tmp_audio_path)
            text_input = self.processor.apply_chat_template(
                messages,
                add_generation_prompt=True,
                tokenize=False
            )

            audio_data, _ = librosa.load(tmp_audio_path, sr=self.sr, mono=True)

            inputs = self.processor(
                text=text_input,
                audios=[audio_data],
                sampling_rate=self.sr,
                return_tensors='pt',
                padding=True,
            ).to(next(self.model.parameters()).device)

            with torch.no_grad():
                generated_ids = self.model.generate(
                    **inputs,
                    max_new_tokens=2048,
                    do_sample=False,
                    repetition_penalty=1.1,
                )

            input_len = inputs['input_ids'].shape[1]
            new_tokens = generated_ids[:, input_len:]
            raw_text = self.processor.batch_decode(
                new_tokens,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=True,
            )[0]

            print(f'   ✅ Inference done [{start_sec:.1f}s-{end_sec:.1f}s]')
            sentiment_data = self._parse_json_response(raw_text)

        finally:
            if os.path.exists(tmp_audio_path):
                os.remove(tmp_audio_path)

        sentiment_data.update({
            'segment_start_sec': round(start_sec, 2),
            'segment_end_sec': round(end_sec, 2),
            'segment_duration_sec': round(end_sec - start_sec, 2)
        })
        return sentiment_data

    def analyze_call(self, preprocessed: dict) -> dict:
        segments = preprocessed['segments']
        print(f'\n🧠 ANALYZING: {Path(preprocessed["original_file"]).name}')
        print(f'   Segments: {len(segments)}')
        print('─' * 50)

        segment_results = []
        for i, (start_sec, end_sec, audio_array) in enumerate(segments):
            print(f'   🔍 Segment {i+1}/{len(segments)} ({start_sec:.1f}s – {end_sec:.1f}s) ... ', end='')
            t0 = time.time()
            result = self.analyze_segment(audio_array, start_sec, end_sec)
            elapsed = time.time() - t0
            print(f"→ {result.get('primary_emotion', 'unknown')} "
                  f"(conf: {result.get('confidence', 0):.2f}) [{elapsed:.1f}s]")
            segment_results.append(result)

        return self._aggregate_segments(segment_results, preprocessed)

    def _aggregate_segments(self, segment_results: list, preprocessed: dict) -> dict:
        valid = [r for r in segment_results if not r.get('parse_error')]
        if not valid:
            return {'error': 'All segments failed to parse', 'segment_details': segment_results}

        def _get_emotion_scores(r):
            return r.get('emotion_scores') or r.get('emotion_score') or r.get('emotions') or {}

        # Average emotion scores
        avg_scores = {}
        for label in self.EMOTION_LABELS:
            scores = [_get_emotion_scores(r).get(label, 0.0) for r in valid]
            avg_scores[label] = round(sum(scores) / len(scores), 3) if scores else 0.0

        dominant_emotion = max(avg_scores, key=avg_scores.get) if avg_scores else 'unknown'

        # Sentiment arc
        sentiment_arc = [
            {
                'time': f"{r.get('segment_start_sec',0):.1f}s–{r.get('segment_end_sec',0):.1f}s",
                'primary_emotion': r.get('primary_emotion', 'unknown'),
                'sentiment': r.get('overall_sentiment', 'unknown'),
                'score': r.get('sentiment_score', 0),
            } for r in segment_results
        ]

        # Final average score
        scores_list = [r.get('sentiment_score', 0) for r in valid if isinstance(r.get('sentiment_score'), (int, float))]
        avg_sentiment_score = round(sum(scores_list) / len(scores_list), 3) if scores_list else 0.0

        # Sarcasm detection
        sarcasm_detected = any(
            r.get('sarcasm_indicators', {}).get('detected', False)
            and r.get('sarcasm_indicators', {}).get('confidence', 0) > 0.5
            for r in valid
        )

        # Escalation Risk
        risk_map = {'low': 0, 'medium': 1, 'high': 2}
        def _infer_risk(r):
            explicit = r.get('call_quality_indicators', {}).get('escalation_risk')
            if explicit in risk_map: return explicit
            scores = _get_emotion_scores(r)
            hot_score = max(scores.get('angry', 0), scores.get('frustrated', 0), scores.get('irritated', 0))
            return 'high' if hot_score >= 0.75 else 'medium' if hot_score >= 0.45 else 'low'

        risk_levels = [_infer_risk(r) for r in valid]
        highest_risk = max(risk_levels, key=lambda x: risk_map.get(x, 0)) if risk_levels else 'low'

        return {
            'metadata': {
                'source_file': str(preprocessed['original_file']),
                'analyzed_at': datetime.now().isoformat(),
                'model_used': self.model_id,
                'call_duration_sec': preprocessed['final_duration_sec'],
                'num_segments_analyzed': len(segment_results),
            },
            'call_summary': {
                'dominant_emotion': dominant_emotion,
                'overall_sentiment': 'positive' if avg_sentiment_score > 0.2 else 'negative' if avg_sentiment_score < -0.2 else 'neutral',
                'sentiment_score': avg_sentiment_score,
                'average_emotion_scores': avg_scores,
                'sarcasm_detected': sarcasm_detected,
                'highest_escalation_risk': highest_risk,
                'sentiment_arc': sentiment_arc,
            },
            'segment_details': segment_results,
        }

print('✅ SentimentAnalyzer defined.')

✅ SentimentAnalyzer defined.


## 🔧 STEP 6b: Verify GPU Placement.

In [21]:


# ── START: GPU placement verification ────────────────────────────────────────

import gc
gc.collect()
torch.cuda.empty_cache()

vram_used  = torch.cuda.memory_allocated() / 1e9
vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9

print('🔍 Model placement check:')
param_devices = set()
for name, param in list(model.named_parameters())[:5]:
    print(f'   {name}: {param.device} | dtype: {param.dtype}')
    param_devices.add(str(param.device))

print(f'\nGPU memory: {vram_used:.2f} GB used / {vram_total:.1f} GB total')
print(f'Available : {vram_total - vram_used:.2f} GB free')

if 'cuda:0' in param_devices or any('cuda' in d for d in param_devices):
    print('\n✅ Model is on GPU — ready for inference!')
else:
    print('\n⚠️  Model not on GPU. Forcing move...')
    model = model.to('cuda:0')
    model.eval()
    print(f'   GPU after move: {torch.cuda.memory_allocated()/1e9:.2f} GB used')

print('\n✅ GPU verification complete.')
# ── END: GPU placement verification ──────────────────────────────────────────

🔍 Model placement check:
   audio_tower.conv1.weight: cuda:0 | dtype: torch.float16
   audio_tower.conv1.bias: cuda:0 | dtype: torch.float16
   audio_tower.conv2.weight: cuda:0 | dtype: torch.float16
   audio_tower.conv2.bias: cuda:0 | dtype: torch.float16
   audio_tower.embed_positions.weight: cuda:0 | dtype: torch.float16

GPU memory: 6.26 GB used / 15.6 GB total
Available : 9.38 GB free

✅ Model is on GPU — ready for inference!

✅ GPU verification complete.


In [22]:


# ── START: emergency GPU fix (run only if inference throws CUDA device errors) ─

import torch, gc
gc.collect()
torch.cuda.empty_cache()
print(f"GPU before: {torch.cuda.memory_allocated()/1e9:.2f} GB used")

for name, param in list(model.named_parameters())[:5]:
    print(f"  {name}: {param.device}")

device = torch.device("cuda:0")
model  = model.to(device)
model.eval()

print(f"\nGPU after: {torch.cuda.memory_allocated()/1e9:.2f} GB used")
print(f"Model device: {next(model.parameters()).device}")
# ── END: emergency GPU fix ────────────────────────────────────────────────────

GPU before: 6.26 GB used
  audio_tower.conv1.weight: cuda:0
  audio_tower.conv1.bias: cuda:0
  audio_tower.conv2.weight: cuda:0
  audio_tower.conv2.bias: cuda:0
  audio_tower.embed_positions.weight: cuda:0

GPU after: 6.26 GB used
Model device: cuda:0


#BRIDGE FUNCTION


In [23]:
# ======================== Remote Diarization Client ========================
# Remote Diarization Client -> Fetches speaker segments from a remote API
# ||
# ||
# ||
# Functions/Methods -> get_remote_diarization() -> Main execution flow
# ||                 |
# ||                 |---> CONFIG.get() -> Retrieve diarizer endpoint
# ||                 |---> requests.get() -> Ping bridge health check
# ||                 |---> json() -> Parse health data
# ||                 |---> requests.post() -> Send file path for processing (with retry)
# ||                 |---> time.sleep() -> Wait before retry on connection drop
# ||                 |---> json() -> Parse speaker turns
# ||                 |
# ||                 |---> Logic Flow -> API communication pipeline:
# ||                                  |
# ||                                  |--- IF URL is missing or default
# ||                                  |    └── Return None
# ||                                  |--- requests.get() -> Verify server health
# ||                                  |--- IF server loading/unreachable
# ||                                  |    └── Return None / Warn user
# ||                                  |--- RETRY LOOP (up to 3 attempts)
# ||                                  |    |--- requests.post() -> Send path to /diarize
# ||                                  |    |--- IF request successful
# ||                                  |    |    └── Return speaker segments
# ||                                  |    |--- IF ConnectionError -> Retry after 5s
# ||                                  |    |--- IF Timeout -> Retry after 5s
# ||                                  |    └── IF HTTPError or Exception -> Return None
# ||                                  |--- IF all retries fail -> Return None
# ||
# ======================================================================

# ---------------------------------------------------------------
# SECTION: IMPORTS
# ---------------------------------------------------------------
import requests
import time

# ---------------------------------------------------------------
# SECTION: FUNCTIONS
# ---------------------------------------------------------------
def get_remote_diarization(local_pc_audio_path):
    url = CONFIG.get("diarizer_url", "").strip()
    if not url or "YOUR_NGROK_URL_HERE" in url:
        print("❌ diarizer_url is not set in CONFIG.")
        print("   Update CONFIG['diarizer_url'] with your ngrok URL from the terminal.")
        return None

    # ── START: health check ───────────────────────────────────────────────────
    health_url = url.replace("/diarize", "/health")
    try:
        h = requests.get(health_url, timeout=10)
        if h.status_code == 200:
            hdata = h.json()
            if not hdata.get("pipeline_loaded"):
                print("⏳ Server is up but pipeline still loading — wait 30s and retry.")
                return None
            print(f"✅ Bridge healthy: device={hdata.get('device','?')}")
        else:
            print(f"⚠️  /health returned {h.status_code} — proceeding anyway")
    except requests.exceptions.ConnectionError:
        print(f"❌ Cannot reach server at {health_url}")
        print("   Is newserver.py running? Is ngrok still connected?")
        return None
    except Exception as he:
        print(f"⚠️  /health check failed ({he}) — proceeding anyway")
    # ── END: health check ─────────────────────────────────────────────────────

    # ── START: post with retry loop ───────────────────────────────────────────
    # ngrok drops connection mid-response on unstable networks
    # retry up to 3 times with 5s wait between attempts
    payload     = {"file_path": local_pc_audio_path}
    max_retries = 3

    for attempt in range(max_retries):
        try:
            print(f"📡 Sending to diarizer (attempt {attempt+1}/{max_retries}): {local_pc_audio_path}")
            response = requests.post(url, json=payload, timeout=600)
            response.raise_for_status()
            data     = response.json()
            segments = data["segments"]
            print(f"✅ Received {len(segments)} speaker turns | "
                  f"{data.get('num_speakers','?')} unique speakers")
            return segments

        except requests.exceptions.ConnectionError as e:
            # ngrok dropped connection — wait and retry
            print(f"⚠️  Connection dropped (attempt {attempt+1}/{max_retries}): {e}")
            if attempt < max_retries - 1:
                print(f"   Retrying in 5s...")
                time.sleep(5)
            else:
                print("❌ All retries failed — connection keeps dropping.")
                print("   Check your internet or restart newserver.py.")
                return None

        except requests.exceptions.Timeout:
            # server is still processing — wait longer and retry
            print(f"⚠️  Request timed out (attempt {attempt+1}/{max_retries})")
            if attempt < max_retries - 1:
                print(f"   Retrying in 5s...")
                time.sleep(5)
            else:
                print("❌ All retries timed out — audio may be too long.")
                return None

        except requests.exceptions.HTTPError as e:
            # HTTP error (4xx/5xx) — no point retrying
            try:
                detail = response.json().get("detail", response.text[:300])
            except Exception:
                detail = response.text[:300]
            print(f"❌ Diarization Bridge HTTP Error: {e}")
            print(f"   Server says: {detail}")
            return None

        except Exception as e:
            # unexpected error — no point retrying
            print(f"❌ Diarization Bridge Error: {type(e).__name__}: {e}")
            return None
    # ── END: post with retry loop ─────────────────────────────────────────────

#STEP 6 : AUDIO UPLOAD & RUN ANALYSIS

In [24]:
# ======================== Audio Upload & Run Analysis ========================
# Audio Upload & Run Analysis -> Orchestrates upload, diarization, and sentiment pipeline.
# ||
# ||
# ||
# Functions/Methods -> files.upload() -> Trigger file input
# ||                 |
# ||                 |---> AudioPreprocessor() -> Init preprocessor
# ||                 |---> SentimentAnalyzer() -> Init analyzer
# ||                 |---> os.path.exists() -> Check file presence
# ||                 |---> os.listdir() -> Search directory
# ||                 |---> get_remote_diarization() -> Fetch speaker IDs + transcripts
# ||                 |---> preprocess() -> Format & segment audio
# ||                 |---> librosa.load() -> Load raw audio mono for Qwen
# ||                 |---> merge_speaker_segments() -> Merge short turns per speaker
# ||                 |---> analyze_segment() -> Inference per speaker chunk (audio only)
# ||                 |---> _aggregate_segments() -> Merge segment data
# ||                 |---> analyze_call() -> Multi-segment inference fallback
# ||                 |---> os.path.join() -> Build output path
# ||                 |---> json.dump() -> Save results to JSON
# ||                 |---> traceback.print_exc() -> Output error logs
# ||
# NOTE: Transcripts from Whisper are attached to output AFTER sentiment analysis.
#       Qwen never sees transcript text — sentiment is purely audio/prosody based.
# ======================================================================

# ---------------------------------------------------------------
# SECTION: IMPORTS
# ---------------------------------------------------------------
from google.colab import files
from pathlib import Path
import os, json, time, traceback

# ---------------------------------------------------------------
# SECTION: MAIN / EXECUTION ENTRY POINT
# ---------------------------------------------------------------
print('📁 Click "Choose Files" to upload your audio file(s):')
uploaded       = files.upload()
uploaded_files = list(uploaded.keys())
print(f'\n✅ Uploaded {len(uploaded_files)} file(s):')
for f in uploaded_files:
    print(f'   📎 {f} ({len(uploaded[f])/1024:.1f} KB)')

assert 'model'     in dir(), '❌ model not defined — run Step 4 first!'
assert 'processor' in dir(), '❌ processor not defined — run Step 4 first!'

preprocessor = AudioPreprocessor(CONFIG)
analyzer     = SentimentAnalyzer(model, processor, CONFIG)
all_results  = {}

# ---------------------------------------------------------------
# SECTION: FUNCTIONS
# ---------------------------------------------------------------
def merge_speaker_segments(segments, min_duration=5.0):
    """
    Merge consecutive turns of the same speaker until chunk reaches min_duration.
    Transcripts from all merged sub-segments are concatenated and stored separately.
    Transcripts are NOT used for sentiment analysis — only attached to output JSON.

    Returns: list of {start, end, speaker, duration, transcript} dicts
    """
    print(f"\n⟶  START merge_speaker_segments()  [segments={len(segments)}, min_duration={min_duration}s]")

    if not segments:
        print("⟵  END   merge_speaker_segments()  [empty — nothing to merge]\n")
        return []

    merged  = []
    current = {
        'start'      : segments[0]['start'],
        'end'        : segments[0]['end'],
        'speaker'    : segments[0]['speaker'],
        # collect transcript text from sub-segments — for output display only
        '_transcripts': [segments[0].get('transcript', {}).get('text', '').strip()],
    }

    for seg in segments[1:]:
        if seg['speaker'] == current['speaker'] and (current['end'] - current['start']) < min_duration:
            current['end'] = seg['end']
            # append transcript of this sub-segment
            t = seg.get('transcript', {}).get('text', '').strip()
            if t:
                current['_transcripts'].append(t)
        else:
            current['duration']   = round(current['end'] - current['start'], 3)
            # join all sub-segment transcripts into one string
            current['transcript'] = ' '.join(current['_transcripts']).strip()
            del current['_transcripts']
            merged.append(current)
            current = {
                'start'      : seg['start'],
                'end'        : seg['end'],
                'speaker'    : seg['speaker'],
                '_transcripts': [seg.get('transcript', {}).get('text', '').strip()],
            }

    current['duration']   = round(current['end'] - current['start'], 3)
    current['transcript'] = ' '.join(current['_transcripts']).strip()
    del current['_transcripts']
    merged.append(current)

    print(f"⟵  END   merge_speaker_segments()  [merged={len(merged)} chunks returned]\n")
    return merged


# ---------------------------------------------------------------
# SECTION: PROCESSING LOOP
# ---------------------------------------------------------------
for audio_filename in uploaded_files:
    print(f'\n{"="*60}')
    print(f'📞 Processing: {audio_filename}')
    print('='*60)
    print(f"\n⟶  START processing loop  [file={audio_filename}]")

    audio_path = f'/content/{audio_filename}'

    if not os.path.exists(audio_path):
        print(f'⚠️  Not found at {audio_path}, searching...')
        matches = [f for f in os.listdir('/content') if audio_filename in f]
        if matches:
            audio_path = f'/content/{matches[0]}'
            print(f'   Found at: {audio_path}')
        else:
            print(f'❌ Could not locate file. Skipping.')
            print(f"⟵  END   processing loop  [skipped — file not found: {audio_filename}]\n")
            continue

    try:
        pc_file_path     = f"C:\\Users\\Pcc\\Downloads\\EMOTION DETECTION NEW\\{audio_filename}"
        speaker_metadata = get_remote_diarization(pc_file_path)
        # expose for QualityScorer in quality dashboard cell
        _last_speaker_metadata = speaker_metadata

        preprocessed = preprocessor.preprocess(audio_path)

        if speaker_metadata:
            print(f"🎙️ Running Speaker-Wise Sentiment Analysis...")
            print(f"   ⚠️  NOTE: Sentiment uses audio/prosody only — transcripts attached post-analysis")

            full_audio, _ = librosa.load(
                audio_path, sr=CONFIG['target_sample_rate'], mono=True
            )
            full_duration = len(full_audio) / CONFIG['target_sample_rate']
            print(f"   Full audio: {full_duration:.1f}s")
            print(f"   Diarization turns: {len(speaker_metadata)}")

            # merge short turns — transcripts collected inside merge function
            merged = merge_speaker_segments(speaker_metadata, min_duration=6.0)
            print(f"   After merging: {len(merged)} speaker chunks")
            for m in merged:
                print(f"     {m['speaker']}: {m['start']:.1f}s → {m['end']:.1f}s ({m['duration']:.1f}s)")
                if m.get('transcript'):
                    print(f"       📝 \"{m['transcript'][:80]}{'...' if len(m['transcript']) > 80 else ''}\"")

            segment_results = []
            for seg in merged:
                print(f"\n⟶  START segment loop  [speaker={seg['speaker']}, {seg['start']:.1f}s → {seg['end']:.1f}s]")

                start_samp  = int(seg['start'] * CONFIG['target_sample_rate'])
                end_samp    = min(int(seg['end'] * CONFIG['target_sample_rate']), len(full_audio))
                audio_chunk = full_audio[start_samp:end_samp]

                if len(audio_chunk) < (4.0 * CONFIG['target_sample_rate']):
                    print(f"   ⏭️  Skipping {seg['speaker']} — too short ({len(audio_chunk)/CONFIG['target_sample_rate']:.1f}s)")
                    print(f"⟵  END   segment loop  [skipped — too short: {seg['speaker']}]\n")
                    continue

                print(f"\n  🎤 {seg['speaker']} ({seg['start']:.1f}s–{seg['end']:.1f}s, {seg['duration']:.1f}s)")

                # ── SENTIMENT ANALYSIS — audio only, NO transcript passed to Qwen ──
                result = analyzer.analyze_segment(audio_chunk, seg['start'], seg['end'])
                if result.get('parse_error') or result.get('primary_emotion') == 'unknown':
                    print(f"   ⚠️  Retrying segment...")
                    result = analyzer.analyze_segment(audio_chunk, seg['start'], seg['end'])

                # ── ATTACH TRANSCRIPT AFTER ANALYSIS — display only ───────────────
                result['speaker']          = seg['speaker']
                result['segment_duration'] = seg['duration']
                result['transcript']       = seg.get('transcript', '')  # ← attached here, never used above
                segment_results.append(result)

                print(f"⟵  END   segment loop  [speaker={seg['speaker']} | emotion={result.get('primary_emotion', 'N/A')} | conf={result.get('confidence', 0):.2f}]\n")

            # aggregate all speaker segment results into single call summary
            results = analyzer._aggregate_segments(segment_results, preprocessed)

            # build per-speaker emotion breakdown with transcript per turn
            speaker_emotion_map = {}
            for r in segment_results:
                spk = r.get('speaker', 'unknown')
                if spk not in speaker_emotion_map:
                    speaker_emotion_map[spk] = []
                speaker_emotion_map[spk].append({
                    'time'           : f"{r.get('segment_start_sec', 0):.1f}s–{r.get('segment_end_sec', 0):.1f}s",
                    'primary_emotion': r.get('primary_emotion', 'unknown'),
                    'sentiment_score': r.get('sentiment_score', 0),
                    'confidence'     : r.get('confidence', 0),
                    'transcript'     : r.get('transcript', ''),  # ← what they said
                })
            results['speaker_emotion_breakdown'] = speaker_emotion_map

            print(f"\n  📊 Speaker emotion breakdown:")
            for spk, turns in speaker_emotion_map.items():
                for t in turns:
                    print(f"     {spk} [{t['time']}] → {t['primary_emotion']} (conf: {t['confidence']:.2f})")
                    if t.get('transcript'):
                        print(f"       📝 \"{t['transcript'][:80]}\"")

        else:
            print("⚠️ Bridge failed. Falling back to default segmentation.")
            results = analyzer.analyze_call(preprocessed)

        stem             = Path(audio_filename).stem
        output_json_path = os.path.join(CONFIG["output_dir"], f"{stem}_sentiment.json")
        with open(output_json_path, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2, ensure_ascii=False)

        all_results[audio_filename] = results

        summary = results.get('call_summary', {})
        print(f'\n{"─"*50}')
        print(f'📊 RESULTS: {audio_filename}')
        print(f'   🎭 Dominant emotion  : {summary.get("dominant_emotion", "N/A")}')
        print(f'   💬 Overall sentiment : {summary.get("overall_sentiment", "N/A")} '
              f'(score: {summary.get("sentiment_score", 0):.3f})')
        print(f'   🎯 Sarcasm detected  : {summary.get("sarcasm_detected", False)}')
        print(f'   ⚠️  Escalation risk   : {summary.get("highest_escalation_risk", "N/A")}')
        print(f'   💾 Saved to          : {output_json_path}')

        print(f"⟵  END   processing loop  [done: {audio_filename} | emotion={summary.get('dominant_emotion', 'N/A')}]\n")

    except Exception as e:
        print(f'\n❌ Failed: {e}')
        traceback.print_exc()
        print(f"⟵  END   processing loop  [exception: {type(e).__name__} | file={audio_filename}]\n")
        all_results[audio_filename] = {'error': str(e), 'file': audio_filename}

print(f'\n✅ Done! Processed {len(all_results)}/{len(uploaded_files)} file(s).')

📁 Click "Choose Files" to upload your audio file(s):


Saving satisfiedsample2.wav to satisfiedsample2.wav

✅ Uploaded 1 file(s):
   📎 satisfiedsample2.wav (4868.1 KB)

📞 Processing: satisfiedsample2.wav

⟶  START processing loop  [file=satisfiedsample2.wav]
✅ Bridge healthy: device=cpu
📡 Sending to diarizer (attempt 1/3): C:\Users\Pcc\Downloads\EMOTION DETECTION NEW\satisfiedsample2.wav
✅ Received 9 speaker turns | 3 unique speakers

🎛️  PREPROCESSING: satisfiedsample2.wav
──────────────────────────────────────────────────
  📊 Loaded: 26.0s | 16000Hz | mono
  🔇 Noise reduction SKIPPED (disabled to preserve original duration)
  🔊 Normalization SKIPPED (disabled to preserve original duration)
  ✂️  Silence trim SKIPPED (disabled to preserve original duration)
  📍 Segment 1: 0.0s – 25.0s
  📍 Segment 2: 23.0s – 26.0s
  💾 Cleaned audio saved: /content/sentiment_results/satisfiedsample2_cleaned.wav
  ✅ Done: 2 segment(s)
🎙️ Running Speaker-Wise Sentiment Analysis...
   ⚠️  NOTE: Sentiment uses audio/prosody only — transcripts attached post-anal

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


   ✅ Inference done [0.7s-6.7s]

🔍 RAW MODEL OUTPUT (first 400 chars):
{'primary_emotion': 'angry', 'confidence': 0.73,'secondary_emotions': ['frustrated'], 'emotion_scores': {'angry': 0.73, 'frustrated': 0.41}, 'vocal_characteristics': {'tone': 'cold', 'pace': 'fast', 'pitch_variation': 'high', 'energy_level': 'high','stress_detected': False, 'hesitations_detected': False, 'voice_breaks_detected': False, 'exaggerated.politeness': True}, 'call_quality_indicators': {'
⟵  END   segment loop  [speaker=SPEAKER_01 | emotion=angry | conf=0.73]


⟶  START segment loop  [speaker=SPEAKER_00, 6.7s → 15.7s]

  🎤 SPEAKER_00 (6.7s–15.7s, 9.0s)


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


   ✅ Inference done [6.7s-15.7s]

🔍 RAW MODEL OUTPUT (first 400 chars):
{'primary_emotion': 'happy', 'confidence': 0.73,'secondary_emotions': ['satisfied'], 'emotion_scores': {'happy': 0.73}, 'vocal_characteristics': {'tone': '温暖的', 'pace': '正常的速度', 'pitch_variation': '平稳的', 'energy_level': '中等的','stress_detected': False, 'hesitations_detected': False, 'voice_breaks_detected': False, 'exaggerated.politeness': False}, 'call_quality_indicators': {'engagement_level': '高'
⟵  END   segment loop  [speaker=SPEAKER_00 | emotion=happy | conf=0.73]


⟶  START segment loop  [speaker=SPEAKER_02, 15.7s → 20.3s]

  🎤 SPEAKER_02 (15.7s–20.3s, 4.6s)


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


   ✅ Inference done [15.7s-20.3s]

🔍 RAW MODEL OUTPUT (first 400 chars):
{'primary_emotion': 'happy', 'confidence': 0.73,'secondary_emotions': ['polite'], 'emotion_scores': {'happy': 0.73}, 'vocal_characteristics': {'tone': '温暖', 'pace': '正常', 'pitch_variation': '平稳', 'energy_level': '中等','stress_detected': False, 'hesitations_detected': False, 'voice_breaks_detected': False, 'exaggerated.politeness': False}, 'call_quality_indicators': {'engagement_level': '高', 'escala
⟵  END   segment loop  [speaker=SPEAKER_02 | emotion=happy | conf=0.73]


⟶  START segment loop  [speaker=SPEAKER_00, 20.3s → 26.0s]

  🎤 SPEAKER_00 (20.3s–26.0s, 5.7s)
   ✅ Inference done [20.3s-26.0s]

🔍 RAW MODEL OUTPUT (first 400 chars):
{'primary_emotion': 'angry', 'confidence': 0.73,'secondary_emotions': ['frustrated'], 'emotion_scores': {'angry': 0.73, 'frustrated': 0.68}, 'vocal_characteristics': {'tone': 'aggressive', 'pace': 'fast', 'pitch_variation': 'high', 'energy_level': 'high','stress_detected': True, 'hes

## 📤 STEP 7: View & Download Results

In [25]:
# ======================== Display Results ========================
# Display Results -> Retrieves and prints the final JSON output of the analysis.
# ||
# ||
# ||
# Functions/Methods -> all_results.get() -> Fetch analysis from memory
# ||                 |
# ||                 |---> json.dumps() -> Format dictionary to JSON string
# ||                 |---> print() -> Display output to console
# ||                 |
# ||                 |---> Logic Flow -> Display pipeline:
# ||                                  |
# ||                                  |--- IF all_results is empty
# ||                                  |    └── print() -> Show missing results error
# ||                                  |--- ELSE
# ||                                  |    ├── Get first filename from uploaded_files
# ||                                  |    ├── all_results.get() -> Fetch result dictionary
# ||                                  |    ├── IF result is None
# ||                                  |    │   └── print() -> Show not found error
# ||                                  |    ├── ELIF 'error' in result
# ||                                  |    │   └── print() -> Show processing failure error
# ||                                  |    └── ELSE
# ||                                  |        └── json.dumps() -> Pretty-print and display JSON
# ||
# ======================================================================

# ---------------------------------------------------------------
# SECTION: MAIN / EXECUTION ENTRY POINT
# ---------------------------------------------------------------
if not all_results:
    print('❌ No results — run Step 6 first.')
else:
    filename_to_show = uploaded_files[0]
    result = all_results.get(filename_to_show)
    if result is None:
        print(f'❌ No result found for {filename_to_show}')
    elif 'error' in result:
        print(f'❌ Processing failed: {result["error"]}')
    else:
        print(f'📋 FULL RESULT: {filename_to_show}')
        print('═' * 60)
        print(json.dumps(result, indent=2))

📋 FULL RESULT: satisfiedsample2.wav
════════════════════════════════════════════════════════════
{
  "metadata": {
    "source_file": "/content/satisfiedsample2.wav",
    "analyzed_at": "2026-03-25T17:57:33.513517",
    "model_used": "Qwen/Qwen2-Audio-7B-Instruct",
    "call_duration_sec": 25.96,
    "num_segments_analyzed": 4
  },
  "call_summary": {
    "dominant_emotion": "happy",
    "overall_sentiment": "neutral",
    "sentiment_score": 0.0,
    "average_emotion_scores": {
      "happy": 0.365,
      "satisfied": 0.0,
      "neutral": 0.0,
      "confused": 0.0,
      "not_interested": 0.0,
      "bored": 0.0,
      "irritated": 0.0,
      "frustrated": 0.273,
      "angry": 0.365,
      "sarcastic": 0.0,
      "anxious": 0.0,
      "sad": 0.0,
      "excited": 0.0,
      "polite_but_cold": 0.0
    },
    "sarcasm_detected": true,
    "highest_escalation_risk": "high",
    "sentiment_arc": [
      {
        "time": "0.7s\u20136.7s",
        "primary_emotion": "angry",
        "sen

In [26]:
# ======================== Download Results ========================
# Download Results -> Triggers browser downloads for generated JSON reports.
# ||
# ||
# ||
# Functions/Methods -> Path().stem -> Extract filename without extension
# ||                 |
# ||                 |---> os.path.join() -> Build output file path
# ||                 |---> os.path.exists() -> Verify file existence
# ||                 |---> colab_files.download() -> Trigger browser download
# ||                 |---> print() -> Log download status
# ||                 |
# ||                 |---> Logic Flow -> Batch download pipeline:
# ||                                  |
# ||                                  |--- Loop through uploaded_files
# ||                                  |    ├── Path().stem & os.path.join() -> Build path
# ||                                  |    ├── IF os.path.exists() is True
# ||                                  |    │   ├── colab_files.download() -> Save to local
# ||                                  |    │   └── print() -> Log success
# ||                                  |    └── ELSE
# ||                                  |        └── print() -> Warn not found
# ||
# ======================================================================

# ---------------------------------------------------------------
# SECTION: IMPORTS
# ---------------------------------------------------------------
from google.colab import files as colab_files

# ---------------------------------------------------------------
# SECTION: MAIN / EXECUTION ENTRY POINT
# ---------------------------------------------------------------
print('📥 Downloading result JSON files...')
for audio_filename in uploaded_files:
    stem      = Path(audio_filename).stem
    json_path = os.path.join(CONFIG['output_dir'], f'{stem}_sentiment.json')
    if os.path.exists(json_path):
        colab_files.download(json_path)
        print(f'  ✅ Downloaded: {stem}_sentiment.json')
    else:
        print(f'  ❌ Not found: {json_path}')

📥 Downloading result JSON files...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  ✅ Downloaded: satisfiedsample2_sentiment.json


In [13]:
# ======================== QualityScorer ========================
# QualityScorer -> Computes per-segment, per-speaker, and call-level
#                  quality/confidence metrics across all three pipeline layers:
#                  diarization, transcription, and emotion detection.
# ||
# || Metrics surfaced:
# ||   Diarization  -> segment duration, short-turn rate, gap ratio,
# ||                   diarization_confidence (PyAnnote score if available)
# ||   Transcription-> avg_logprob, no_speech_prob, compression_ratio,
# ||                   transcription_confidence (derived 0–1)
# ||   Emotion      -> Qwen confidence, emotion score entropy (decisiveness),
# ||                   sarcasm confidence, parse_error rate
# ||
# ======================================================================

# ---------------------------------------------------------------
# SECTION: IMPORTS
# ---------------------------------------------------------------
import numpy as np
from collections import defaultdict

# ---------------------------------------------------------------
# SECTION: CLASSES
# ---------------------------------------------------------------
class QualityScorer:
    """
    Computes confidence and quality metrics for every layer of the pipeline.

    Usage:
        scorer  = QualityScorer()
        quality = scorer.score(results, speaker_metadata)

    Returns a dict with:
        per_segment_quality   : list — one entry per analysed segment turn
        per_speaker_quality   : dict — one entry per speaker ID
        call_level_quality    : dict — aggregate call quality summary
        layer_scores          : dict — one 0–1 score per layer (diarization,
                                        transcription, emotion)
        overall_pipeline_score: float — single 0–1 score for the whole call
    """

    # ── Entropy thresholds ────────────────────────────────────────────────────
    # A uniform distribution over 14 emotions has entropy = log2(14) ≈ 3.81 bits
    # If entropy is close to max, the model was confused (all emotions equal)
    # If entropy is near 0, the model was very decisive (one emotion dominates)
    MAX_EMOTION_ENTROPY = np.log2(14)

    def score(self, results: dict, speaker_metadata: list = None) -> dict:
        """
        Main entry point.

        Args:
            results          : the full dict returned by analyzer._aggregate_segments()
                               (same dict saved to JSON — contains segment_details)
            speaker_metadata : raw diarization segments from get_remote_diarization()
                               (list of dicts with start, end, speaker,
                                diarization_confidence, transcript)
                               Pass None if diarization was skipped.

        Returns:
            quality report dict (see class docstring)
        """
        segment_details = results.get("segment_details", [])

        # ── START: score each analysed segment ───────────────────────────────
        per_segment = []
        for seg in segment_details:
            per_segment.append(self._score_segment(seg))
        # ── END: score each analysed segment ─────────────────────────────────

        # ── START: score diarization layer ───────────────────────────────────
        diarization_score, diarization_details = self._score_diarization(
            speaker_metadata, results
        )
        # ── END: score diarization ────────────────────────────────────────────

        # ── START: score transcription layer ─────────────────────────────────
        transcription_score, transcription_details = self._score_transcription(
            speaker_metadata
        )
        # ── END: score transcription ──────────────────────────────────────────

        # ── START: score emotion layer ────────────────────────────────────────
        emotion_score, emotion_details = self._score_emotion(segment_details)
        # ── END: score emotion ────────────────────────────────────────────────

        # ── START: per-speaker quality rollup ────────────────────────────────
        per_speaker = self._per_speaker_quality(per_segment)
        # ── END: per-speaker quality ──────────────────────────────────────────

        # ── START: overall pipeline score ────────────────────────────────────
        # Weighted: diarization 20%, transcription 30%, emotion 50%
        # (emotion is the hardest and most important layer for the product)
        overall = round(
            0.20 * diarization_score +
            0.30 * transcription_score +
            0.50 * emotion_score,
            3
        )
        # ── END: overall score ────────────────────────────────────────────────

        # ── START: call-level flags ───────────────────────────────────────────
        call_flags = []
        if diarization_score  < 0.5: call_flags.append("diarization_unreliable")
        if transcription_score < 0.5: call_flags.append("transcription_unreliable")
        if emotion_score       < 0.5: call_flags.append("emotion_detection_unreliable")
        if overall             < 0.4: call_flags.append("low_overall_pipeline_confidence")
        # ── END: call-level flags ─────────────────────────────────────────────

        return {
            "per_segment_quality"   : per_segment,
            "per_speaker_quality"   : per_speaker,
            "call_level_quality"    : {
                "total_segments_analysed"  : len(segment_details),
                "segments_with_parse_error": emotion_details["parse_error_count"],
                "segments_skipped_too_short": diarization_details.get("short_segment_count", 0),
                "call_flags"               : call_flags,
            },
            "layer_scores": {
                "diarization"  : {"score": diarization_score,   "details": diarization_details},
                "transcription": {"score": transcription_score,  "details": transcription_details},
                "emotion"      : {"score": emotion_score,        "details": emotion_details},
            },
            "overall_pipeline_score": overall,
        }

    # ── PRIVATE: score one analysed segment ──────────────────────────────────
    def _score_segment(self, seg: dict) -> dict:
        speaker  = seg.get("speaker", "unknown")
        start    = seg.get("segment_start_sec", seg.get("start", 0))
        end      = seg.get("segment_end_sec",   seg.get("end",   0))
        duration = seg.get("segment_duration_sec", seg.get("duration", end - start))

        # ── transcription confidence (from Whisper quality block) ─────────────
        tq = seg.get("transcription_quality", {})
        if not tq:
            # try nested inside transcript dict (older format)
            tq = seg.get("transcript", {}).get("transcription_quality", {})
        trans_conf = tq.get("transcription_confidence", None)
        trans_flags = tq.get("quality_flags", [])

        # ── emotion confidence ────────────────────────────────────────────────
        emotion_conf = seg.get("confidence", None)

        # ── emotion entropy — how decisive was Qwen? ──────────────────────────
        emotion_entropy = None
        entropy_norm    = None
        es = seg.get("emotion_scores", {})
        if es:
            vals = np.array(list(es.values()), dtype=float)
            vals = vals / (vals.sum() + 1e-9)  # normalise to probability dist
            # Shannon entropy: H = -sum(p * log2(p))
            nonzero = vals[vals > 1e-9]
            raw_entropy = float(-np.sum(nonzero * np.log2(nonzero)))
            emotion_entropy = round(raw_entropy, 3)
            # Normalise: 0 = fully decisive, 1 = fully uncertain
            entropy_norm = round(raw_entropy / self.MAX_EMOTION_ENTROPY, 3)

        # ── emotion decisiveness: 1 - normalised_entropy ─────────────────────
        emotion_decisiveness = round(1.0 - entropy_norm, 3) if entropy_norm is not None else None

        # ── segment-level flags ───────────────────────────────────────────────
        seg_flags = list(trans_flags)  # copy transcription flags
        if seg.get("parse_error"):
            seg_flags.append("emotion_parse_error")
        if emotion_conf is not None and emotion_conf < 0.4:
            seg_flags.append("low_emotion_confidence")
        if emotion_decisiveness is not None and emotion_decisiveness < 0.4:
            seg_flags.append("emotion_model_uncertain")
        if trans_conf is not None and trans_conf < 0.4:
            seg_flags.append("low_transcription_confidence")

        return {
            "speaker"              : speaker,
            "time"                 : f"{start:.1f}s–{end:.1f}s",
            "duration_sec"         : round(duration, 2),
            "primary_emotion"      : seg.get("primary_emotion", "unknown"),
            "transcript_preview"   : (seg.get("transcript") or "")[:80],
            "emotion_confidence"   : round(emotion_conf, 3) if emotion_conf is not None else None,
            "emotion_decisiveness" : emotion_decisiveness,
            "emotion_entropy_bits" : emotion_entropy,
            "transcription_confidence": trans_conf,
            "transcription_flags"  : trans_flags,
            "segment_flags"        : seg_flags,
        }

    # ── PRIVATE: diarization layer score ─────────────────────────────────────
    def _score_diarization(self, speaker_metadata: list, results: dict) -> tuple:
        if not speaker_metadata:
            return 0.5, {"note": "diarization_skipped_or_unavailable"}

        total        = len(speaker_metadata)
        short_count  = sum(1 for s in speaker_metadata
                           if (s.get("end", 0) - s.get("start", 0)) < 1.5)
        short_rate   = round(short_count / total, 3) if total else 0

        # diarization_confidence from PyAnnote (None if not available)
        conf_vals = [
            s["diarization_confidence"]
            for s in speaker_metadata
            if s.get("diarization_confidence") is not None
        ]
        avg_diar_conf = round(float(np.mean(conf_vals)), 3) if conf_vals else None

        # gap ratio: fraction of call duration that is unlabelled
        if speaker_metadata:
            call_start = speaker_metadata[0].get("start", 0)
            call_end   = speaker_metadata[-1].get("end",   0)
            call_span  = call_end - call_start
            labelled   = sum(
                (s.get("end", 0) - s.get("start", 0)) for s in speaker_metadata
            )
            gap_ratio  = round(max(0, call_span - labelled) / call_span, 3) if call_span > 0 else 0
        else:
            gap_ratio = 0

        # derive score
        # penalise high short-segment rate and high gap ratio
        short_penalty = short_rate * 0.4
        gap_penalty   = gap_ratio  * 0.3
        base_score    = 1.0 - short_penalty - gap_penalty
        if avg_diar_conf is not None:
            # blend with PyAnnote's own confidence
            base_score = 0.6 * base_score + 0.4 * avg_diar_conf

        score = round(float(np.clip(base_score, 0.0, 1.0)), 3)

        details = {
            "total_raw_segments"       : total,
            "short_segment_count"      : short_count,
            "short_segment_rate"       : short_rate,
            "gap_ratio"                : gap_ratio,
            "avg_pyannote_confidence"  : avg_diar_conf,
            "pyannote_confidence_note" : (
                "Available — PyAnnote track scores returned"
                if conf_vals else
                "Not available on this PyAnnote version (None returned)"
            ),
        }
        return score, details

    # ── PRIVATE: transcription layer score ───────────────────────────────────
    def _score_transcription(self, speaker_metadata: list) -> tuple:
        if not speaker_metadata:
            return 0.5, {"note": "no_diarization_data"}

        # collect transcription_quality blocks from raw diarization metadata
        tq_blocks = []
        for seg in speaker_metadata:
            tq = seg.get("transcript", {})
            if isinstance(tq, dict):
                q = tq.get("transcription_quality", {})
                if q:
                    tq_blocks.append(q)

        if not tq_blocks:
            return 0.5, {"note": "no_transcription_quality_data_in_response"}

        confs = [b["transcription_confidence"] for b in tq_blocks
                 if b.get("transcription_confidence") is not None]
        logprobs = [b["avg_logprob"] for b in tq_blocks
                    if b.get("avg_logprob") is not None]
        no_speech = [b["no_speech_prob"] for b in tq_blocks
                     if b.get("no_speech_prob") is not None]

        avg_conf     = round(float(np.mean(confs)),    3) if confs     else 0.5
        avg_logprob  = round(float(np.mean(logprobs)), 3) if logprobs  else None
        avg_no_speech= round(float(np.mean(no_speech)),3) if no_speech else None

        # count segments with flags
        all_flags = []
        for b in tq_blocks:
            all_flags.extend(b.get("quality_flags", []))
        flag_counts = defaultdict(int)
        for f in all_flags:
            flag_counts[f] += 1

        details = {
            "segments_with_quality_data": len(tq_blocks),
            "avg_transcription_confidence": avg_conf,
            "avg_whisper_logprob"        : avg_logprob,
            "avg_no_speech_prob"         : avg_no_speech,
            "flag_counts"                : dict(flag_counts),
        }
        return avg_conf, details

    # ── PRIVATE: emotion layer score ──────────────────────────────────────────
    def _score_emotion(self, segment_details: list) -> tuple:
        if not segment_details:
            return 0.5, {"note": "no_segments"}

        valid = [s for s in segment_details if not s.get("parse_error")]
        parse_error_count = len(segment_details) - len(valid)

        confs = [s["confidence"] for s in valid
                 if isinstance(s.get("confidence"), (int, float))]

        # entropy over each segment's emotion_scores
        entropies = []
        for s in valid:
            es = s.get("emotion_scores", {})
            if es:
                vals = np.array(list(es.values()), dtype=float)
                vals = vals / (vals.sum() + 1e-9)
                nz   = vals[vals > 1e-9]
                entropies.append(float(-np.sum(nz * np.log2(nz))))

        avg_conf    = round(float(np.mean(confs)),     3) if confs     else 0.5
        avg_entropy = round(float(np.mean(entropies)), 3) if entropies else None
        # decisiveness: low entropy = high decisiveness
        avg_decisiveness = (
            round(1.0 - avg_entropy / self.MAX_EMOTION_ENTROPY, 3)
            if avg_entropy is not None else None
        )

        # combined: blend confidence with decisiveness
        if avg_decisiveness is not None:
            score = round(0.6 * avg_conf + 0.4 * avg_decisiveness, 3)
        else:
            score = avg_conf

        details = {
            "total_segments"        : len(segment_details),
            "valid_segments"        : len(valid),
            "parse_error_count"     : parse_error_count,
            "parse_error_rate"      : round(parse_error_count / len(segment_details), 3),
            "avg_emotion_confidence": avg_conf,
            "avg_emotion_entropy_bits": avg_entropy,
            "avg_emotion_decisiveness": avg_decisiveness,
        }
        return score, details

    # ── PRIVATE: per-speaker quality rollup ──────────────────────────────────
    def _per_speaker_quality(self, per_segment: list) -> dict:
        by_speaker = defaultdict(list)
        for seg in per_segment:
            by_speaker[seg["speaker"]].append(seg)

        result = {}
        for spk, segs in by_speaker.items():
            ec = [s["emotion_confidence"]    for s in segs if s["emotion_confidence"]    is not None]
            ed = [s["emotion_decisiveness"]  for s in segs if s["emotion_decisiveness"]  is not None]
            tc = [s["transcription_confidence"] for s in segs if s["transcription_confidence"] is not None]
            all_flags = []
            for s in segs:
                all_flags.extend(s.get("segment_flags", []))

            result[spk] = {
                "turns_analysed"              : len(segs),
                "avg_emotion_confidence"      : round(float(np.mean(ec)),  3) if ec else None,
                "avg_emotion_decisiveness"    : round(float(np.mean(ed)),  3) if ed else None,
                "avg_transcription_confidence": round(float(np.mean(tc)),  3) if tc else None,
                "flag_summary"                : dict(defaultdict(int, {f: all_flags.count(f)
                                                                        for f in set(all_flags)})),
                "per_turn_detail"             : segs,
            }
        return result


# ---------------------------------------------------------------
# SECTION: MAIN / EXECUTION ENTRY POINT
# ---------------------------------------------------------------
print("✅ QualityScorer defined.")


✅ QualityScorer defined.


In [14]:
# ======================== Quality Dashboard ========================
# Quality Dashboard -> Computes and visualises pipeline accuracy/confidence
#                      metrics across diarization, transcription, and emotion
#                      detection layers.
# ||
# || Run AFTER Step 6 (Audio Upload & Run Analysis).
# || Requires: all_results, speaker_metadata (from get_remote_diarization)
# ======================================================================

# ---------------------------------------------------------------
# SECTION: IMPORTS
# ---------------------------------------------------------------
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import numpy as np
from pathlib import Path
from collections import defaultdict

matplotlib.rcParams["figure.dpi"] = 150

# ---------------------------------------------------------------
# SECTION: MAIN / EXECUTION ENTRY POINT
# ---------------------------------------------------------------
if not all_results:
    print("❌ No results found — run Step 6 first!")
else:
    # find first valid result
    result, chosen_file = None, None
    for fname, res in all_results.items():
        if "error" not in res and "segment_details" in res:
            result, chosen_file = res, fname
            break

    if result is None:
        print("❌ No valid results to score.")
    else:
        # ── START: run quality scorer ─────────────────────────────────────────
        # 'speaker_metadata' is the raw list from get_remote_diarization()
        # It is set in the processing loop in Step 6 — check if it exists
        raw_diar = None
        try:
            raw_diar = locals().get("_last_speaker_metadata") or globals().get("_last_speaker_metadata")  # set in Step 6 loop
        except NameError:
            print("⚠️  speaker_metadata not in scope — diarization scores will be partial")

        scorer  = QualityScorer()
        quality = scorer.score(result, raw_diar)

        # save quality report alongside sentiment JSON
        stem        = Path(chosen_file).stem
        quality_path = f"{CONFIG['output_dir']}/{stem}_quality.json"
        with open(quality_path, "w", encoding="utf-8") as f:
            # serialise numpy floats
            import json as _json
            class _NpEncoder(_json.JSONEncoder):
                def default(self, obj):
                    if isinstance(obj, (np.integer,)): return int(obj)
                    if isinstance(obj, (np.floating,)): return float(obj)
                    if isinstance(obj, np.ndarray): return obj.tolist()
                    return super().default(obj)
            _json.dump(quality, f, indent=2, ensure_ascii=False, cls=_NpEncoder)
        print(f"💾 Quality report saved → {quality_path}")
        # ── END: run quality scorer ───────────────────────────────────────────

        # ── START: print readable summary ────────────────────────────────────
        ls  = quality["layer_scores"]
        ops = quality["overall_pipeline_score"]
        clq = quality["call_level_quality"]

        print(f"\n{'='*58}")
        print(f"📊  PIPELINE QUALITY REPORT  |  {Path(chosen_file).name}")
        print(f"{'='*58}")
        print(f"  🔵 Diarization   score : {ls['diarization']['score']:.3f}   "
              f"({_conf_label(ls['diarization']['score'])})")
        print(f"  🟡 Transcription score : {ls['transcription']['score']:.3f}  "
              f"({_conf_label(ls['transcription']['score'])})")
        print(f"  🟣 Emotion detect score: {ls['emotion']['score']:.3f}   "
              f"({_conf_label(ls['emotion']['score'])})")
        print(f"  {'─'*50}")
        print(f"  ⭐ Overall pipeline    : {ops:.3f}   ({_conf_label(ops)})")
        if clq.get("call_flags"):
            print(f"\n  ⚠️  Flags: {', '.join(clq['call_flags'])}")
        print(f"\n  Segments analysed   : {clq['total_segments_analysed']}")
        print(f"  Parse errors        : {clq['segments_with_parse_error']}")

        print(f"\n  Per-speaker summary:")
        for spk, sq in quality["per_speaker_quality"].items():
            print(f"    {spk} ({sq['turns_analysed']} turns)")
            ec = sq['avg_emotion_confidence']
            ed = sq['avg_emotion_decisiveness']
            tc = sq['avg_transcription_confidence']
            if ec  is not None: print(f"      Emotion confidence  : {ec:.3f}  ({_conf_label(ec)})")
            if ed  is not None: print(f"      Emotion decisiveness: {ed:.3f}  ({_conf_label(ed)})")
            if tc  is not None: print(f"      Transcript confidence: {tc:.3f}  ({_conf_label(tc)})")
            if sq["flag_summary"]:
                print(f"      Flags               : {sq['flag_summary']}")

        print(f"\n  Per-turn detail:")
        for seg in quality["per_segment_quality"]:
            flag_str = f"  ⚠️ {seg['segment_flags']}" if seg["segment_flags"] else ""
            ec  = seg['emotion_confidence']
            tc  = seg['transcription_confidence']
            ed  = seg['emotion_decisiveness']
            print(
                f"    {seg['speaker']} [{seg['time']}]  "
                f"emotion={seg['primary_emotion']} "
                f"(conf={ec:.2f if ec else 'N/A'}, decisive={ed:.2f if ed else 'N/A'})  "
                f"transcript_conf={tc:.2f if tc else 'N/A'}"
                f"{flag_str}"
            )
            if seg.get("transcript_preview"):
                print(f"      📝 \"{seg['transcript_preview']}\"")
        print(f"{'='*58}\n")
        # ── END: print summary ────────────────────────────────────────────────

        # ── START: build quality dashboard chart ──────────────────────────────
        DARK_BG, PANEL_BG = "#121212", "#1E1E1E"
        TEXT_COL, GRID_COL = "#E0E0E0", "#333333"
        COLORS = {
            "diarization"  : "#64B5F6",   # blue
            "transcription": "#FFD54F",   # amber
            "emotion"      : "#CE93D8",   # purple
            "overall"      : "#80CBC4",   # teal
        }

        fig = plt.figure(figsize=(14, 18), facecolor=DARK_BG)
        gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.55, wspace=0.35,
                                top=0.93, bottom=0.06, left=0.10, right=0.97)

        def _style(ax, title):
            ax.set_facecolor(PANEL_BG)
            for sp in ax.spines.values():
                sp.set_color(GRID_COL); sp.set_linewidth(1.2)
            ax.tick_params(colors=TEXT_COL, labelsize=9)
            ax.set_title(title, color=TEXT_COL, fontsize=11, fontweight="bold", pad=10)
            ax.yaxis.label.set_color(TEXT_COL); ax.yaxis.label.set_fontsize(10)
            ax.xaxis.label.set_color(TEXT_COL); ax.xaxis.label.set_fontsize(10)

        # ── Panel 1: Layer scores gauge (top-left) ────────────────────────────
        ax0 = fig.add_subplot(gs[0, 0])
        layers  = ["Diarization", "Transcription", "Emotion\nDetection", "Overall\nPipeline"]
        l_scores= [
            ls["diarization"]["score"],
            ls["transcription"]["score"],
            ls["emotion"]["score"],
            ops,
        ]
        l_colors= [COLORS["diarization"], COLORS["transcription"],
                   COLORS["emotion"],     COLORS["overall"]]
        bars = ax0.barh(layers, l_scores, color=l_colors, edgecolor="none", height=0.55)
        ax0.set_xlim(0, 1.15)
        for bar, val in zip(bars, l_scores):
            ax0.text(val + 0.02, bar.get_y() + bar.get_height()/2,
                     f"{val:.2f}  {_conf_label(val)}",
                     va="center", color=TEXT_COL, fontsize=9)
        ax0.axvline(0.7, color="#555", linestyle="--", linewidth=1, alpha=0.6)
        ax0.axvline(0.4, color="#FF6B6B", linestyle="--", linewidth=1, alpha=0.5)
        ax0.invert_yaxis()
        ax0.xaxis.grid(True, color=GRID_COL, linestyle="--", linewidth=0.7, alpha=0.6)
        ax0.set_axisbelow(True)
        ax0.set_xlabel("Confidence Score (0–1)")
        _style(ax0, "🎯  Layer Confidence Scores")

        # ── Panel 2: Per-speaker emotion confidence (top-right) ───────────────
        ax1 = fig.add_subplot(gs[0, 1])
        spk_data = quality["per_speaker_quality"]
        spk_names = list(spk_data.keys())
        spk_ec    = [spk_data[s]["avg_emotion_confidence"]    or 0 for s in spk_names]
        spk_ed    = [spk_data[s]["avg_emotion_decisiveness"]  or 0 for s in spk_names]
        spk_tc    = [spk_data[s]["avg_transcription_confidence"] or 0 for s in spk_names]
        x = np.arange(len(spk_names))
        w = 0.25
        ax1.bar(x - w, spk_ec, w, label="Emotion conf",     color=COLORS["emotion"],       edgecolor="none")
        ax1.bar(x,     spk_ed, w, label="Emotion decisive",  color="#F48FB1",               edgecolor="none")
        ax1.bar(x + w, spk_tc, w, label="Transcript conf",  color=COLORS["transcription"], edgecolor="none")
        ax1.set_xticks(x); ax1.set_xticklabels(spk_names, color=TEXT_COL)
        ax1.set_ylim(0, 1.15)
        ax1.yaxis.grid(True, color=GRID_COL, linestyle="--", linewidth=0.7, alpha=0.6)
        ax1.set_axisbelow(True)
        ax1.legend(facecolor=PANEL_BG, edgecolor=GRID_COL,
                   labelcolor=TEXT_COL, fontsize=8, loc="upper right")
        ax1.axhline(0.7, color="#555", linestyle="--", linewidth=1, alpha=0.5)
        _style(ax1, "🎤  Per-Speaker Quality")

        # ── Panel 3: Per-segment emotion confidence timeline (middle, full width)
        ax2 = fig.add_subplot(gs[1, :])
        seg_q  = quality["per_segment_quality"]
        s_idx  = list(range(len(seg_q)))
        s_ec   = [s["emotion_confidence"]    or 0 for s in seg_q]
        s_ed   = [s["emotion_decisiveness"]  or 0 for s in seg_q]
        s_tc   = [s["transcription_confidence"] or 0 for s in seg_q]
        s_spk  = [s["speaker"]   for s in seg_q]
        s_time = [s["time"]      for s in seg_q]

        spk_set     = sorted(set(s_spk))
        spk_palette = ["#64B5F6", "#FFD54F", "#A5D6A7", "#FF8A65"]
        spk_color   = {s: spk_palette[i % len(spk_palette)] for i, s in enumerate(spk_set)}

        ax2.plot(s_idx, s_ec, color=COLORS["emotion"],       linewidth=2,   label="Emotion conf",    zorder=4)
        ax2.plot(s_idx, s_ed, color="#F48FB1",               linewidth=1.5, label="Emotion decisive",zorder=3, alpha=0.8)
        ax2.plot(s_idx, s_tc, color=COLORS["transcription"], linewidth=1.5, label="Transcript conf", zorder=3, linestyle="--", alpha=0.85)

        for i, (ec, spk) in enumerate(zip(s_ec, s_spk)):
            ax2.scatter(i, ec, color=spk_color[spk], s=60, zorder=5, edgecolors=DARK_BG, linewidths=1.0)

        # flag low-confidence turns
        for i, seg in enumerate(seg_q):
            if seg.get("segment_flags"):
                ax2.axvspan(i - 0.4, i + 0.4, alpha=0.12, color="#FF6B6B", zorder=1)

        ax2.set_xticks(s_idx)
        ax2.set_xticklabels(
            [f"{s['speaker'].replace('SPEAKER_','S')}\n{s['time']}" for s in seg_q],
            rotation=35, ha="right", fontsize=7.5
        )
        ax2.set_ylim(-0.05, 1.15)
        ax2.axhline(0.7, color="#555", linestyle="--", linewidth=1, alpha=0.5)
        ax2.axhline(0.4, color="#FF6B6B", linestyle="--", linewidth=0.8, alpha=0.45)
        ax2.set_ylabel("Confidence (0–1)")
        ax2.yaxis.grid(True, color=GRID_COL, linestyle="--", linewidth=0.7, alpha=0.6)
        ax2.set_axisbelow(True)

        # speaker color legend
        spk_patches = [mpatches.Patch(color=spk_color[s], label=s) for s in spk_set]
        leg1 = ax2.legend(handles=spk_patches, title="Speaker", facecolor=PANEL_BG,
                          edgecolor=GRID_COL, labelcolor=TEXT_COL, fontsize=8,
                          title_fontsize=8, loc="upper right")
        leg2 = ax2.legend(facecolor=PANEL_BG, edgecolor=GRID_COL, labelcolor=TEXT_COL,
                          fontsize=8, loc="upper left")
        ax2.add_artist(leg1)
        _style(ax2, "📈  Per-Segment Confidence Timeline  (red shading = flagged turn)")

        # ── Panel 4: Diarization details (bottom-left) ───────────────────────
        ax3 = fig.add_subplot(gs[2, 0])
        dd   = ls["diarization"]["details"]
        d_labels = ["Short seg\nrate", "Gap\nratio", "Diarization\nscore"]
        d_vals   = [
            dd.get("short_segment_rate", 0),
            dd.get("gap_ratio",          0),
            ls["diarization"]["score"],
        ]
        d_colors = ["#FF8A65", "#FF8A65", COLORS["diarization"]]
        ax3.bar(d_labels, d_vals, color=d_colors, edgecolor="none", width=0.5)
        ax3.set_ylim(0, 1.1)
        for i, v in enumerate(d_vals):
            ax3.text(i, v + 0.02, f"{v:.3f}", ha="center", color=TEXT_COL, fontsize=9)
        ax3.yaxis.grid(True, color=GRID_COL, linestyle="--", linewidth=0.7, alpha=0.6)
        ax3.set_axisbelow(True)
        pyannote_note = (
            f"PyAnnote conf: {dd.get('avg_pyannote_confidence','N/A')}"
            if dd.get("avg_pyannote_confidence") else "PyAnnote conf: N/A"
        )
        ax3.set_xlabel(pyannote_note)
        _style(ax3, "🔵  Diarization Quality")

        # ── Panel 5: Transcription details (bottom-right) ────────────────────
        ax4 = fig.add_subplot(gs[2, 1])
        td   = ls["transcription"]["details"]
        t_labels = ["Avg\nlogprob\n(norm)", "Avg no-\nspeech prob", "Transcript\nscore"]
        raw_lp   = td.get("avg_whisper_logprob", -0.5) or -0.5
        norm_lp  = float(np.clip((raw_lp + 2.0) / 2.0, 0, 1))
        t_vals   = [
            norm_lp,
            td.get("avg_no_speech_prob", 0.3) or 0.3,
            ls["transcription"]["score"],
        ]
        t_colors = [COLORS["transcription"], "#FF8A65", COLORS["transcription"]]
        ax4.bar(t_labels, t_vals, color=t_colors, edgecolor="none", width=0.5)
        ax4.set_ylim(0, 1.1)
        for i, v in enumerate(t_vals):
            ax4.text(i, v + 0.02, f"{v:.3f}", ha="center", color=TEXT_COL, fontsize=9)
        ax4.yaxis.grid(True, color=GRID_COL, linestyle="--", linewidth=0.7, alpha=0.6)
        ax4.set_axisbelow(True)
        fc  = td.get("flag_counts", {})
        fc_str = ", ".join(f"{k}:{v}" for k, v in fc.items()) if fc else "no flags"
        ax4.set_xlabel(f"flags: {fc_str}", fontsize=7.5)
        _style(ax4, "🟡  Transcription Quality")

        fig.suptitle(
            f"Pipeline Quality Dashboard  |  {Path(chosen_file).name}\n"
            f"Overall score: {ops:.3f}  ({_conf_label(ops)})",
            color=TEXT_COL, fontsize=13, fontweight="bold", y=0.975
        )

        dash_path = f"{CONFIG['output_dir']}/{Path(chosen_file).stem}_quality_dashboard.png"
        fig.savefig(dash_path, dpi=150, bbox_inches="tight", facecolor=DARK_BG)
        plt.show()
        print(f"\n✅ Quality dashboard saved → {dash_path}")
        # ── END: build quality dashboard ──────────────────────────────────────


# ── helper: label a 0–1 score ────────────────────────────────────────────────
def _conf_label(v):
    if v is None: return "N/A"
    if v >= 0.80: return "✅ High"
    if v >= 0.60: return "🟡 Moderate"
    if v >= 0.40: return "🟠 Low"
    return "🔴 Unreliable"


❌ No valid results to score.


In [15]:
# ======================== Sentiment Visualization Dashboard ========================
# Sentiment Visualization Dashboard -> Generates 2-panel emotion and sentiment arc chart
# ||
# ||
# ||
# Functions/Methods -> %matplotlib inline -> Set plot backend
# ||                 |
# ||                 |---> plt.figure() -> Init canvas
# ||                 |---> gridspec.GridSpec() -> Define vertical layout
# ||                 |---> ax1.barh() -> Plot horizontal bars
# ||                 |---> ax2.fill_between() -> Shade polarity zones
# ||                 |---> ax2.plot() -> Draw timeline arc
# ||                 |---> fig.savefig() -> Save to PNG
# ||                 |---> plt.show() -> Render in notebook
# ||                 |
# ||                 |---> Logic Flow -> Plotting execution:
# ||                                  |
# ||                                  |--- IF no results -> Show error
# ||                                  |--- ELSE
# ||                                  |    ├── Find valid result with 'call_summary'
# ||                                  |    ├── IF no valid result -> Show error
# ||                                  |    └── ELSE
# ||                                  |        ├── Extract summary, arc, scores
# ||                                  |        ├── Init canvas & GridSpec
# ||                                  |        ├── Define styling & colors
# ||                                  |        ├── ax1.barh() -> Render bar chart
# ||                                  |        ├── IF arc data exists
# ||                                  |        │   ├── ax2.fill_between() -> Color zones
# ||                                  |        │   └── ax2.plot() & ax2.scatter() -> Plot timeline
# ||                                  |        ├── Apply axis styling
# ||                                  |        ├── fig.savefig() -> Export PNG
# ||                                  |        └── plt.show() -> Display
# ||
# ======================================================================

# ---------------------------------------------------------------
# SECTION: IMPORTS
# ---------------------------------------------------------------
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['figure.dpi'] = 150  # Increased DPI for crisper charts
import matplotlib.gridspec as gridspec
import numpy as np
from pathlib import Path

# ---------------------------------------------------------------
# SECTION: MAIN / EXECUTION ENTRY POINT
# ---------------------------------------------------------------
if not all_results:
    print("❌  No results found — run Step 6 first!")
else:
    result, chosen_file = None, None
    for fname, res in all_results.items():
        if "error" not in res and "call_summary" in res:
            result, chosen_file = res, fname
            break

    if result is None:
        print("❌  No valid results to visualise.")
    else:
        summary  = result["call_summary"]
        segments = result.get("segment_details", [])
        meta     = result.get("metadata", {})

        EMOTION_LABELS = [
            "happy", "satisfied", "neutral", "confused",
            "not_interested", "bored", "irritated",
            "frustrated", "angry", "sarcastic",
            "anxious", "sad", "excited", "polite_but_cold",
        ]

        # Enhanced color palette for better contrast and modern look
        EMOTION_COLORS = {
            "happy": "#FFD700", "satisfied": "#98FB98", "neutral": "#B0C4DE",
            "confused": "#DDA0DD", "not_interested": "#808080", "bored": "#A9A9A9",
            "irritated": "#FFA07A", "frustrated": "#FF8C00", "angry": "#FF4500",
            "sarcastic": "#BA55D3", "anxious": "#FF69B4", "sad": "#4169E1",
            "excited": "#00FFFF", "polite_but_cold": "#708090",
        }

        avg_scores = summary.get("average_emotion_scores", {})
        scores_arr = np.array([avg_scores.get(e, 0.0) for e in EMOTION_LABELS])

        arc      = summary.get("sentiment_arc", [])
        arc_x    = list(range(1, len(arc) + 1))
        arc_vals = [s.get("score", 0) for s in arc]
        arc_emos = [s.get("primary_emotion", "neutral") for s in arc]
        arc_time = [s.get("time", f"Seg {i+1}") for i, s in enumerate(arc)]

        # Refined Dark Mode theme
        DARK_BG, PANEL_BG = "#121212", "#1E1E1E"
        TEXT_COL, GRID_COL, ACCENT = "#E0E0E0", "#333333", "#BB86FC"

        # Taller figure for better spacing
        fig = plt.figure(figsize=(12, 16), facecolor=DARK_BG)
        gs  = gridspec.GridSpec(2, 1, figure=fig, hspace=0.35, top=0.92, bottom=0.08, left=0.15, right=0.95)

        ax1 = fig.add_subplot(gs[0, 0])
        ax2 = fig.add_subplot(gs[1, 0])

        def style_ax(ax, title):
            ax.set_facecolor(PANEL_BG)
            for sp in ax.spines.values():
                sp.set_color(GRID_COL)
                sp.set_linewidth(1.5) # Thicker borders
            ax.tick_params(colors=TEXT_COL, labelsize=10, length=6, width=1.5)
            ax.set_title(title, color=TEXT_COL, fontsize=14, fontweight="bold", pad=15)
            ax.yaxis.label.set_color(TEXT_COL)
            ax.yaxis.label.set_fontsize(12)
            ax.xaxis.label.set_color(TEXT_COL)
            ax.xaxis.label.set_fontsize(12)

        bar_colors = [EMOTION_COLORS[e] for e in EMOTION_LABELS]
        # Added slight rounded effect by removing edge colors and adjusting width
        bars = ax1.barh(EMOTION_LABELS, scores_arr, color=bar_colors, edgecolor="none", height=0.7)
        for bar, val in zip(bars, scores_arr):
            if val > 0.01: # Lowered threshold to show more values
                ax1.text(val + 0.015, bar.get_y() + bar.get_height() / 2, f"{val:.2f}",
                         va="center", ha="left", color=TEXT_COL, fontsize=9, fontweight='medium')

        ax1.set_xlim(0, max(scores_arr) + 0.15 if len(scores_arr) > 0 and max(scores_arr) > 0 else 1.1)
        ax1.set_xlabel("Confidence Score (0 – 1)", color=TEXT_COL)
        style_ax(ax1, "🎭  Emotion Score Distribution")
        ax1.invert_yaxis()
        ax1.xaxis.grid(True, color=GRID_COL, linestyle='--', linewidth=0.8, alpha=0.7)
        ax1.set_axisbelow(True) # Put grid behind bars

        if arc_vals:
            arc_colors = [EMOTION_COLORS.get(e, ACCENT) for e in arc_emos]

            # Smooth gradient-like fills
            ax2.fill_between(arc_x, arc_vals, 0, where=[v >= 0 for v in arc_vals],
                             alpha=0.25, color="#4CAF50", interpolate=True)
            ax2.fill_between(arc_x, arc_vals, 0, where=[v < 0 for v in arc_vals],
                             alpha=0.25, color="#F44336", interpolate=True)

            # Thicker line with markers
            ax2.plot(arc_x, arc_vals, color=ACCENT, linewidth=2.5, zorder=4, alpha=0.8)
            ax2.scatter(arc_x, arc_vals, c=arc_colors, s=100, zorder=5,
                        edgecolors=DARK_BG, linewidths=1.5) # Dark edge for pop

            ax2.axhline(0, color="#666666", linewidth=1.5, linestyle="--", zorder=3)
            ax2.set_ylim(-1.1, 1.1)
            ax2.set_xticks(arc_x)
            ax2.set_xticklabels(arc_time, rotation=35, ha="right", fontsize=9)
            ax2.set_ylabel("Sentiment Polarity (-1 to 1)")
            ax2.yaxis.grid(True, color=GRID_COL, linestyle='--', linewidth=0.8, alpha=0.7)
            ax2.set_axisbelow(True)

        style_ax(ax2, "📈  Sentiment Arc Timeline")

        fig.suptitle(f"Analysis Dashboard | {Path(chosen_file).name}",
                     color=TEXT_COL, fontsize=16, fontweight="bold", y=0.96)

        out_path = f"{CONFIG['output_dir']}/{Path(chosen_file).stem}_dashboard.png"
        fig.savefig(out_path, dpi=150, bbox_inches="tight", facecolor=DARK_BG)
        plt.show()

        print(f"\n✅ Dashboard saved → {out_path}")

❌  No valid results to visualise.
